# Lab 1 · Notebook 4 — Capstone: **TextLab**, a multi-task NLP app ⏱ ~25 min

You now have every ingredient. Assemble them into one app with **five tabs**:

| Tab | Model head | Output component |
|---|---|---|
| Sentiment | sequence classification | `gr.Label` |
| Entities | token classification | `gr.HighlightedText` |
| Fill the blank | masked LM | `gr.Label` |
| Zero-shot | NLI / entailment | `gr.Label` |
| FAQ search *(stretch)* | embeddings | `gr.Dataframe` |

**Rules of the game.** Write each function as a plain, testable Python function (TODO 8–12).
Run the **self-test cell**; it tells you which tabs are ready. Then launch the app and export it as a script.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from functools import lru_cache
import torch, gradio as gr
from transformers import pipeline, AutoTokenizer, AutoModel

DEVICE = 0 if torch.cuda.is_available() else -1
SENT_MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
NER_MODEL  = "dslim/distilbert-NER"
FILL_MODEL = "distilbert/distilbert-base-uncased"
ZS_MODEL   = "typeform/distilbert-base-uncased-mnli"
EMB_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

@lru_cache(maxsize=None)
def get_pipe(task, model, **kwargs):
    return pipeline(task, model=model, device=DEVICE, **kwargs)

@lru_cache(maxsize=None)
def _embedder():
    return AutoTokenizer.from_pretrained(EMB_MODEL), AutoModel.from_pretrained(EMB_MODEL).eval()

def embed(texts):
    """texts -> (n, 384) tensor of unit-length sentence embeddings (same as Notebook 2)."""
    tok, model = _embedder()
    batch = tok(texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        hidden = model(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1)
    return torch.nn.functional.normalize((hidden * mask).sum(1) / mask.sum(1), dim=-1)

FAQ = [
    "You can reset your password from the login page by clicking 'Forgot password'.",
    "Our office is open Monday to Friday from 9am to 5pm.",
    "Refunds are issued to the original payment method within 5 working days.",
    "To change your delivery address, edit it in your account settings before the order ships.",
    "We offer student discounts with a valid university email address.",
    "The mobile app is available for both iOS and Android.",
    "Contact support by email or through the chat widget in the bottom corner.",
    "Your subscription renews automatically at the end of each billing period.",
]

W1022 21:50:03.496000 54519 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Step 1 · The five functions
Each one returns **exactly the format its Gradio component expects** (see the table above and Notebook 3). Empty input → `gr.Error`.

### TODO 8 — sentiment → `{label: probability}`

In [2]:
def analyze_sentiment(text):
    if not text or not text.strip():
        raise gr.Error("Please enter some text.")
    scores = get_pipe("sentiment-analysis", SENT_MODEL)(text, top_k=None)
    return {s["label"]: float(s["score"]) for s in scores}

### TODO 9 — entities → `{"text": ..., "entities": [{"entity", "start", "end"}]}`

In [3]:
def extract_entities(text):
    if not text or not text.strip():
        raise gr.Error("Please enter some text.")
    found = get_pipe("token-classification", NER_MODEL, aggregation_strategy="simple")(text)
    return {"text": text,
            "entities": [{"entity": e["entity_group"], "start": int(e["start"]), "end": int(e["end"])} for e in found]}

### TODO 10 — fill the blank → `{token: probability}` (needs a mask token, `k` between 1 and 10)

In [4]:
def fill_mask(text, k=5):
    fill = get_pipe("fill-mask", FILL_MODEL)
    mask = fill.tokenizer.mask_token
    if not text or mask not in text:
        raise gr.Error(f"Your sentence must contain {mask}")
    k = max(1, min(int(k), 10))
    return {r["token_str"].strip(): float(r["score"]) for r in fill(text, top_k=k)}

### TODO 11 — zero-shot with **user-defined labels**
The user types labels separated by commas (`"billing, shipping, bug"`). Parse them (strip spaces, drop empties, need at least 2),
classify, and return `{label: score}`.

In [5]:
def zero_shot(text, labels_csv):
    labels = [l.strip() for l in (labels_csv or "").split(",") if l.strip()]
    if not text or not text.strip():
        raise gr.Error("Please enter some text.")
    if len(labels) < 2:
        raise gr.Error("Give at least two labels, separated by commas.")
    out = get_pipe("zero-shot-classification", ZS_MODEL)(text, candidate_labels=labels)
    return {l: float(s) for l, s in zip(out["labels"], out["scores"])}

### TODO 12 *(stretch)* — FAQ search → rows for a `gr.Dataframe`
Return `[[rank, sentence, score], ...]` for the top-`k` FAQ entries (use `embed` and `FAQ`; the code from Notebook 2 is your template).

In [6]:
def search_faq(query, k=3):
    if not query or not query.strip():
        raise gr.Error("Please type a question.")
    scores = (embed([query]) @ embed(FAQ).T)[0]
    top = scores.topk(min(int(k), len(FAQ)))
    return [[rank + 1, FAQ[int(i)], round(float(s), 3)] for rank, (s, i) in enumerate(zip(top.values, top.indices))]

## Step 2 · Self-test (run it as often as you like — it shows which tabs are ready)

In [7]:
def run_tests():
    def t_sent():
        r = analyze_sentiment("I absolutely loved it, wonderful!")
        assert set(r) == {"POSITIVE", "NEGATIVE"} and abs(sum(r.values()) - 1) < 1e-3, r
        assert r["POSITIVE"] > 0.9, "a clearly positive review should score > 0.9"
    def t_ner():
        r = extract_entities("Angela Merkel visited Google in Zurich.")
        kinds = {e["entity"] for e in r["entities"]}
        assert r["text"] and {"PER", "ORG", "LOC"} <= kinds, kinds
        assert all(isinstance(e["start"], int) for e in r["entities"]), "start/end must be plain ints"
    def t_fill():
        r = fill_mask("Paris is the [MASK] of France.", 3)
        assert len(r) == 3 and "capital" in r, r
        for bad in ("no blank here", ""):
            try: fill_mask(bad, 3)
            except gr.Error: continue
            raise AssertionError("missing [MASK] must raise gr.Error")
    def t_zs():
        r = zero_shot("The app crashes when I upload a photo", "billing, bug report, shipping")
        assert set(r) == {"billing", "bug report", "shipping"}, r
        assert max(r, key=r.get) == "bug report", r
        try: zero_shot("hello", "onlyone")
        except gr.Error: return
        raise AssertionError("fewer than 2 labels must raise gr.Error")
    def t_faq():
        rows = search_faq("I forgot my login credentials", 3)
        assert len(rows) == 3 and rows[0][0] == 1 and "password" in rows[0][1], rows

    for name, fn in [("TODO 8  sentiment", t_sent), ("TODO 9  entities", t_ner), ("TODO 10 fill-mask", t_fill),
                     ("TODO 11 zero-shot", t_zs), ("TODO 12 FAQ search (stretch)", t_faq)]:
        try:
            fn(); status = "✅ pass"
        except NotImplementedError:
            status = "⬜ not written yet"
        except AssertionError as e:
            status = f"❌ {str(e)[:110]}"
        except Exception as e:
            status = f"❌ {type(e).__name__}: {str(e)[:100]}"
        print(f"{name:30s} {status}")

run_tests()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 15050.46it/s]

TODO 8  sentiment              ✅ pass


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 7404.27it/s]

TODO 9  entities               ✅ pass


... (5 outputs left out)

TODO 11 zero-shot              ✅ pass


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8250.51it/s]

TODO 12 FAQ search (stretch)   ✅ pass


## Step 3 · The UI
The wiring is given — read it, then notice how little code it takes once the functions exist. 
**Customise at least one thing** (title, example sentences, a tab description) so the app is yours.

In [8]:
def build_app():
    with gr.Blocks(title="TextLab") as app:
        gr.Markdown("# 🧪 TextLab\nFive NLP tasks powered by small BERT-family models from the Hugging Face Hub.")

        with gr.Tab("Sentiment"):
            t1 = gr.Textbox(lines=4, label="Review")
            o1 = gr.Label(num_top_classes=2, label="Sentiment")
            gr.Button("Analyze", variant="primary").click(analyze_sentiment, t1, o1)
            gr.Examples(["Great value for money!", "The battery died after a week."], inputs=t1)

        with gr.Tab("Entities"):
            t2 = gr.Textbox(lines=4, label="Text")
            o2 = gr.HighlightedText(label="Entities")
            gr.Button("Find entities", variant="primary").click(extract_entities, t2, o2)
            gr.Examples(["Angela Merkel visited the Google office in Zurich last Tuesday."], inputs=t2)

        with gr.Tab("Fill the blank"):
            t3 = gr.Textbox(value="Paris is the [MASK] of France.", label="Sentence with [MASK]")
            k3 = gr.Slider(1, 10, value=5, step=1, label="Guesses")
            o3 = gr.Label(label="Predictions")
            gr.Button("Predict", variant="primary").click(fill_mask, [t3, k3], o3)

        with gr.Tab("Zero-shot"):
            t4 = gr.Textbox(lines=3, label="Text")
            l4 = gr.Textbox(value="billing, bug report, shipping", label="Your labels (comma-separated)")
            o4 = gr.Label(label="Scores")
            gr.Button("Classify", variant="primary").click(zero_shot, [t4, l4], o4)
            gr.Examples([["My parcel never arrived", "billing, bug report, shipping"]], inputs=[t4, l4])

        with gr.Tab("FAQ search"):
            t5 = gr.Textbox(label="Ask a question")
            k5 = gr.Slider(1, 5, value=3, step=1, label="Results")
            o5 = gr.Dataframe(headers=["#", "Answer", "Similarity"], label="Best matches")
            gr.Button("Search", variant="primary").click(search_faq, [t5, k5], o5)
            gr.Examples(["When will I get my money back?", "Is there a cheaper plan for students?"], inputs=t5)
    return app

In [9]:
gr.close_all()
app = build_app()
app.launch()          # add share=True to get a temporary public URL you can open on your phone

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 4 · Ship it: export a standalone script
A notebook is a *lab*; a **script** is something you can deploy, version and hand to a colleague. The cell below writes your functions
into `textlab_app.py`. Then, in a **terminal**: `python textlab_app.py` and open the printed URL.

In [10]:
import inspect
HEADER = '''"""TextLab — exported from Lab 1 (Notebook 4).  Run:  python textlab_app.py"""
import warnings; warnings.filterwarnings("ignore")
from functools import lru_cache
import torch, gradio as gr
from transformers import pipeline, AutoTokenizer, AutoModel

DEVICE = 0 if torch.cuda.is_available() else -1
SENT_MODEL = "%s"
NER_MODEL  = "%s"
FILL_MODEL = "%s"
ZS_MODEL   = "%s"
EMB_MODEL  = "%s"
FAQ = %r
''' % (SENT_MODEL, NER_MODEL, FILL_MODEL, ZS_MODEL, EMB_MODEL, FAQ)

parts = [HEADER] + [inspect.getsource(f) for f in
                    (get_pipe, _embedder, embed, analyze_sentiment, extract_entities, fill_mask, zero_shot, search_faq, build_app)]
parts.append('if __name__ == "__main__":\n    build_app().launch()\n')
with open("textlab_app.py", "w") as fh:
    fh.write("\n\n".join(p.rstrip() + "\n" for p in parts))
print("wrote textlab_app.py  (", sum(1 for _ in open("textlab_app.py")), "lines )")

wrote textlab_app.py  ( 117 lines )


## ✅ Checkpoint — what you built, and where it goes next
A tested, multi-task, user-facing NLP app in about 100 lines. For your **final project** the same skeleton becomes:
*your task + your data + a fair evaluation + a usable interface*. The take-home assignment starts that journey:
**evaluate** these models properly, **extend** the app, and **critique** a model card.

➡️ Open `assignment/ASSIGNMENT.md` and `notebooks/05_homework_starter.ipynb`.